# 🦀 Day 5: bindgen — Auto-Generating Rust Bindings from C Headers
---

Today we use **bindgen** to generate Rust FFI bindings from C headers.

- **What bindgen does**: C header → Rust `extern` declarations + types
- **build.rs integration**: Run at compile time
- **`bindgen::Builder`**: Allowlist, blocklist, options
- **C enums, structs, unions** → Rust equivalents
- **Opaque types**: `struct foo;` → empty struct
- **Safe wrappers**: Hide unsafe behind idiomatic Rust

In [ ]:
// bindgen is typically used in build.rs, not in notebooks.
// Here we show the PATTERN: what the generated code looks like.

// Suppose we have a C header:
//   typedef struct { int x; int y; } Point;
//   int distance(Point a, Point b);

// bindgen would generate something like:
#[repr(C)]
#[derive(Debug, Copy, Clone)]
pub struct Point {
    pub x: std::ffi::c_int,
    pub y: std::ffi::c_int,
}

// extern "C" { pub fn distance(a: Point, b: Point) -> c_int; }  // bindgen output

// For demo, we implement distance ourselves (no real C lib)
fn distance(a: Point, b: Point) -> i32 {
    let dx = (a.x - b.x) as f64;
    let dy = (a.y - b.y) as f64;
    (dx * dx + dy * dy).sqrt() as i32
}

// Safe wrapper over the raw binding
fn safe_distance(a: (i32, i32), b: (i32, i32)) -> i32 {
    let pa = Point { x: a.0, y: a.1 };
    let pb = Point { x: b.0, y: b.1 };
    distance(pa, pb)
}

let p1 = Point { x: 0, y: 0 };
let p2 = Point { x: 3, y: 4 };
println!("distance((0,0), (3,4)) = {}", safe_distance((0, 0), (3, 4)));

## build.rs pattern for bindgen

```rust
// build.rs
fn main() {
    let bindings = bindgen::Builder::default()
        .header("wrapper.h")
        .allowlist_function("my_func")
        .blocklist_type("unwanted_type")
        .generate()
        .expect("bindgen failed");
    bindings.write_to_file("src/bindings.rs")
        .expect("write failed");
}
```

`Cargo.toml`:
```toml
[build-dependencies]
bindgen = "0.69"
```

In [ ]:
// Opaque types: C has "struct Foo;" (forward decl, no definition)
// bindgen generates: pub struct Foo { _private: [u8; 0] }

#[repr(C)]
pub struct OpaqueHandle {
    _private: [u8; 0],
}

// You only ever pass *mut OpaqueHandle or *const OpaqueHandle to C
// Rust never constructs or inspects the contents

// Wrapper types for safety: wrap *mut T in a struct with Drop
pub struct SafeHandle {
    ptr: *mut OpaqueHandle,
}

impl Drop for SafeHandle {
    fn drop(&mut self) {
        // Call C destroy function here
    }
}

## 📝 Exercise: Write safe wrappers for generated bindings

Given bindgen output for `void* create()` and `void destroy(void*)`, design a `Resource` struct that:
- Wraps the raw pointer
- Implements `Drop` to call `destroy`
- Implements a factory `Resource::new() -> Option<Resource>` that returns `None` if `create()` returns null

In [ ]:
// YOUR CODE HERE

## 🎯 Key Takeaways

1. **bindgen** generates Rust FFI bindings from C headers at build time
2. **build.rs** runs bindgen; output goes to `src/bindings.rs` or similar
3. **`Builder`** API: `header()`, `allowlist_function()`, `blocklist_type()`
4. **Opaque types** become zero-sized structs; only pass pointers
5. **Safe wrappers** (RAII, `Drop`) hide unsafe from users

---
**Tomorrow:** Memory layout — repr(C), alignment, size_of